# 33 | Final harmonized COCO evaluation: 3 architectures × 3 seeds (GPU)

**Article 1 — target journal:** *Smart Agricultural Technology* (Elsevier).  
**Goal:** close the primary multi-seed comparison of YOLO11n-seg, YOLO26n-seg and Mask R-CNN on the **already inspected** FaroPigSeg date-disjoint test set.

**No training. No threshold tuning. No new test split.** The notebook reuses archived instance predictions from Experiments **09** (original YOLO11/Mask R-CNN) and **23** (three YOLO26 seeds), and performs inference only for the four new 2027/2028 checkpoints. All nine sets of predictions are evaluated by exactly the same `pycocotools.COCOeval(iouType='segm')` settings.

**Experiment population:** 373 test images, 4,648 annotated pig instances, 5 acquisition dates. These are not five independent farms. The test data have been viewed repeatedly in earlier experiments; the expanded analyses are descriptive and **not** a fresh blind external validation.

**Outputs:** one well-formatted Excel workbook, English scientific figures (600-dpi PNG, vector PDF and SVG), protocol and execution report JSON, and one results ZIP. Full COCO prediction JSON files remain stored in Drive for audit; the ZIP contains their hashes and provenance, not multi-hundred-megabyte duplicate predictions.

**In Colab:** Runtime → Change runtime type → **T4 GPU or L4**. Run cells in order. Retried sessions resume partial prediction batches without redoing completed images.

## 1. Confirm accelerator and install the same evaluation libraries

The notebook intentionally stops on CPU; it does not train any model.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required. Set Colab Runtime → Change runtime type → GPU (T4 or L4).')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
%pip -q install "ultralytics==8.4.182" "pycocotools>=2.0.8,<3" "pyyaml>=6,<7" "pandas>=2,<3" "openpyxl>=3.1,<4" "matplotlib>=3.8" pillow

## 2. Mount Drive and verify frozen dataset / historical COCO protocol

The GT JSON, image IDs, five test dates, hashes of YAML and test lists, and reference COCO settings **must** agree with Experiment 09. The notebook will refuse incompatible inputs.

In [ ]:
from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
from contextlib import redirect_stdout
from IPython.display import display
from PIL import Image
from collections import defaultdict
import hashlib, json, io, os, gc, time, platform, sys, zipfile
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import torch, torchvision, ultralytics
from ultralytics import YOLO
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskutils

assert ultralytics.__version__ == '8.4.182', 'Unexpected Ultralytics version; do not change the inference library silently.'
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'Project folder missing: {ROOT}'
SPLIT=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'
RES=ROOT/'03_Resultados'
RUN=RES/'06_Entrenamiento'
REPORTS=RES/'05_Reportes'
METRICS=RES/'03_Metricas'
EXPORT=ROOT/'05_Exportaciones_ZIP'
OUT=REPORTS/'CERDOS_33_COCO_9_Semillas'
FIGS=RES/'04_Figuras'/'CERDOS_33_COCO_9_Semillas'
for d in (OUT,FIGS,METRICS,EXPORT):d.mkdir(parents=True,exist_ok=True)
YAML=SPLIT/'data_fecha.yaml'
MANIFEST=SPLIT/'particion_fecha_manifest.json'
BASE09=REPORTS/'CERDOS_09_COCO_Armonizado'
REF09=BASE09/'CERDOS_09_Protocolo.json'
GT_FILE=BASE09/'CERDOS_09_GT_test_coco.json'
REPORT09=REPORTS/'CERDOS_09_Armonizacion_COCO.json'
BASE23=REPORTS/'CERDOS_23_YOLO26_COCO_Test'
REF23=BASE23/'CERDOS_23_Protocolo.json'
REPORT23=BASE23/'CERDOS_23_Informe_Evaluacion.json'
for f in (YAML,MANIFEST,REF09,GT_FILE,REPORT09,REF23,REPORT23):
    if not f.is_file():raise FileNotFoundError(f'Archived dependency missing: {f}')

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
def readjson(p):return json.loads(Path(p).read_text(encoding='utf-8'))
def write_atomic(p,obj):
    p=Path(p);t=p.with_suffix(p.suffix+'.writing')
    t.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    os.replace(t,p)

ref09=readjson(REF09);ref23=readjson(REF23)
assert ref09['pipeline']=='CERDOS_09_COCOv1'
assert int(ref09['max_dets'])==100 and int(ref09['yolo_imgsz'])==640
assert float(ref09['yolo_conf'])==0.001 and float(ref09['yolo_iou_nms'])==0.7
assert bool(ref09['yolo_retina_masks']) is True
assert float(ref09['mask_score_thresh'])==0.001 and float(ref09['mask_bin_thresh'])==0.5
cfg=yaml.safe_load(YAML.read_text(encoding='utf-8'))
TEST_TXT=Path(cfg['test'])
assert TEST_TXT.is_file() and sha256(YAML)==ref09['yaml_sha256']
assert sha256(TEST_TXT)==ref09['test_file_sha256']
paths=[Path(line.strip()) for line in TEST_TXT.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(paths)==373 and len(set(map(str,paths)))==373
assert all(p.is_file() for p in paths), 'At least one original test image is missing in Drive.'
manifest=readjson(MANIFEST)['asignaciones_fecha']
assert all(manifest.get(p.name[:10])=='test' for p in paths)
bydate=defaultdict(list)
for image_id,p in enumerate(paths,1):bydate[p.name[:10]].append(image_id)
assert len(bydate)==5
gt_data=readjson(GT_FILE)
assert len(gt_data['images'])==373 and len(gt_data['annotations'])==4648
for image_id,p in enumerate(paths,1):
    im=gt_data['images'][image_id-1]
    assert int(im['id'])==image_id and im['file_name']==p.name, (image_id,p.name)
assert ref23['test_txt_sha256']==sha256(TEST_TXT) and ref23['gt_json_sha256']==sha256(GT_FILE)
assert ref23['yaml_sha256']==sha256(YAML) and ref23['reference_notebook']=='CERDOS_09_COCOv1'
print('Verified: 373 test images / 4,648 instance masks / 5 disjoint test dates.')
print('COCO reference and date split match Experiments 09 and 23.')

## 3. Audit the nine selected checkpoints

Compare stored model hashes against the original evaluation protocol and training-completion manifests. Original seeds 2026 are reused; **only the new seeds 2027/2028** have not yet been evaluated for YOLO11n-seg and Mask R-CNN. The date-held-out test is never used for checkpoint selection.

In [ ]:
SEEDS=(2026,2027,2028)
WEIGHTS={};metadata=[]
for family in ('YOLO11n-seg','YOLO26n-seg','Mask R-CNN'):
    for seed in SEEDS:
        if family=='YOLO11n-seg':
            run=(RUN/'YOLO11n_fecha_v1' if seed==2026 else RUN/'YOLO11n_multisemilla_v1'/f'seed_{seed}')
            best=run/'weights'/'best.pt'
            done=None if seed==2026 else run/'CERDOS_31_Completed.json'
        elif family=='YOLO26n-seg':
            run=RUN/'YOLO26n_fecha_v1'/f'seed_{seed}'
            best=run/'weights'/'best.pt'
            done=run/'CERDOS_22_ENTRENAMIENTO_FINALIZADO.json'
        else:
            run=(RUN/'MaskRCNN_fecha_v1' if seed==2026 else RUN/'MaskRCNN_multisemilla_v1'/f'seed_{seed}')
            best=run/'maskrcnn_best_model.pt'
            done=None if seed==2026 else run/'CERDOS_32_ENTRENAMIENTO_FINALIZADO.json'
        if not best.is_file():raise FileNotFoundError(f'Checkpoint missing: {best}')
        digest=sha256(best)
        if family=='YOLO11n-seg' and seed==2026:
            assert digest==ref09['yolo_sha256'],'Baseline YOLO11 hash differs from Experiment 09.'
        if family=='Mask R-CNN' and seed==2026:
            assert digest==ref09['mask_sha256'],'Baseline Mask R-CNN hash differs from Experiment 09.'
        if family=='YOLO26n-seg':
            assert digest==ref23['weights_sha256'][str(seed)],'YOLO26 differs from Experiment 23.'
        if done is not None:
            if not done.is_file():raise FileNotFoundError(f'Missing completion record: {done}')
            status=readjson(done)
            assert int(status.get('seed',status.get('required',{}).get('seed',status.get('config',{}).get('seed',-1))))==seed
            recorded=status.get('weights_sha256',status.get('best_weights_sha256'))
            assert recorded==digest,f'Completion record hash mismatch: {family} {seed}'
            yaml_record=status.get('yaml_sha256',status.get('config',{}).get('frozen_inputs',{}).get('yaml_sha256'))
            if yaml_record:assert yaml_record==sha256(YAML),f'Checkpoint dataset mismatch: {family} {seed}'
        WEIGHTS[(family,seed)]=best
        metadata.append({'Model':family,'Seed':seed,'Checkpoint':str(best),'SHA256':digest,
                         'Source experiment':{'YOLO11n-seg':3 if seed==2026 else 31,
                          'YOLO26n-seg':22,'Mask R-CNN':6 if seed==2026 else 32}[family]})
print(pd.DataFrame(metadata)[['Model','Seed','Source experiment']].to_string(index=False))
assert len(metadata)==9

## 4. Protect the already archived predictions (5 runs)

No re-inference for the three YOLO26 seeds or the original YOLO11 and Mask R-CNN models. Every 2026/YOLO26 prediction is accepted only after confirming the historical protocol, original model hash, and associated SHA-256 checks wherever those checks were archived.

In [ ]:
PRED_ARCHIVE={
    ('YOLO11n-seg',2026):BASE09/'CERDOS_09_Pred_YOLO_coco.json',
    ('Mask R-CNN',2026):BASE09/'CERDOS_09_Pred_MaskRCNN_coco.json'
}
for seed in SEEDS:
    PRED_ARCHIVE[('YOLO26n-seg',seed)]=BASE23/f'CERDOS_23_Pred_seed_{seed}.json'

# Explicit checks of Experiment 23 hash manifests, not merely file existence.
for seed in SEEDS:
    path=PRED_ARCHIVE[('YOLO26n-seg',seed)]
    lock=BASE23/f'CERDOS_23_Pred_seed_{seed}_hash.json'
    assert path.is_file() and lock.is_file(),f'Archived YOLO26 seed {seed} missing.'
    j=readjson(lock)
    assert j['weights_sha256']==sha256(WEIGHTS[('YOLO26n-seg',seed)])
    assert j['protocolo_sha256']==sha256(REF23)
    assert j['predictions_sha256']==sha256(path)
for p in PRED_ARCHIVE.values():
    assert p.is_file(),f'Missing earlier predictions: {p}'
# Experiment 09 pred files did not include separate prediction SHA manifests;
# freeze their hashes under Experiment 33 and reproduce the original COCO AP before accepting.
HISTORICAL_PRED_SHA={str(k):sha256(p) for k,p in PRED_ARCHIVE.items()}

PROTO={
 'experiment':'CERDOS_33_COCO_9_Seeds_v1',
 'parent_protocol09_sha256':sha256(REF09),'parent_protocol23_sha256':sha256(REF23),
 'ground_truth_sha256':sha256(GT_FILE),'test_txt_sha256':sha256(TEST_TXT),
 'yaml_sha256':sha256(YAML),'manifest_sha256':sha256(MANIFEST),
 'historical_prediction_sha256':HISTORICAL_PRED_SHA,
 'checkpoint_sha256':{f'{m} | {s}':sha256(p) for (m,s),p in WEIGHTS.items()},
 'test_images':373,'test_GT_instances':4648,'test_dates':5,
 'coco_eval':{'iouType':'segm','maxDets':[1,10,100],
              'IoU thresholds':'0.50:0.05:0.95'},
 'inference':{'YOLO11n-seg':{'imgsz':640,'conf':0.001,'iou':0.7,'max_det':100,'retina_masks':True},
              'Mask R-CNN':{'min_size':512,'max_size':768,'score_thresh':0.001,'detections_per_img':100,'mask_bin_thresh':0.5}},
 'selection':'Best checkpoint by validation mask AP; no tuning of thresholds on test.',
 'analysis_status':'post-hoc descriptive; test examined in earlier experiments',
 'no_external_validation':True,
}
PROTO_FILE=OUT/'CERDOS_33_Protocolo.json'
if PROTO_FILE.exists():
    assert readjson(PROTO_FILE)==PROTO,'Experiment 33 protocol changed: never mix cached predictions.'
else:
    if any(OUT.glob('CERDOS_33_Pred_*.json')) or any(OUT.glob('CERDOS_33_Partial_*.json')):
        raise RuntimeError('Uncontrolled old prediction cache without protocol; inspect and resolve it first.')
    write_atomic(PROTO_FILE,PROTO)
PROTO_HASH=sha256(PROTO_FILE)
print('Frozen Experiment 33 protocol SHA-256:',PROTO_HASH[:16])

## 5. Infer only the four missing models; resume safely after Colab disconnects

For each seed, predictions are saved incrementally every 25 images. Completed predictions have a digest lock. If interrupted, rerun from the beginning: the notebook checks the lock and resumes at the first unsaved image. Both detectors use **exactly the original Experiment 09 preprocessing and thresholds**, not newly optimized operating points.

In [ ]:
def mask_to_rle(mask):
    rle=maskutils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle['counts']=rle['counts'].decode('ascii')
    return rle

def new_maskrcnn():
    model=maskrcnn_resnet50_fpn(weights=None,weights_backbone=None,
               min_size=512,max_size=768,trainable_backbone_layers=3)
    box_input=model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor=FastRCNNPredictor(box_input,2)
    mask_input=model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor=MaskRCNNPredictor(mask_input,256,2)
    model.roi_heads.score_thresh=0.001
    model.roi_heads.detections_per_img=100
    return model

def validate_rows(rows):
    assert isinstance(rows,list) and len(rows)>0
    for r in rows:
        assert 1<=int(r['image_id'])<=373 and int(r['category_id'])==1
        assert 0<=float(r['score'])<=1 and 'segmentation' in r
        assert 'size' in r['segmentation'] and 'counts' in r['segmentation']
    return True

def new_predictions(family,seed):
    slug={'YOLO11n-seg':'YOLO11','Mask R-CNN':'MaskRCNN'}[family]
    stem=f'CERDOS_33_Pred_{slug}_seed_{seed}'
    path=OUT/(stem+'.json');lock=OUT/(stem+'_hash.json');partial=OUT/(stem+'_partial.json')
    expected_wsha=sha256(WEIGHTS[(family,seed)])
    if path.exists():
        if not lock.is_file():raise RuntimeError(f'Prediction file without hash lock: {path}')
        meta=readjson(lock)
        assert meta['protocol_sha256']==PROTO_HASH and meta['weights_sha256']==expected_wsha
        assert meta['predictions_sha256']==sha256(path)
        if partial.exists():raise RuntimeError(f'Final and partial files coexist for {stem}; inspect manually.')
        rows=readjson(path);validate_rows(rows)
        print(f'Reusing audited {stem}: {len(rows)} COCO predictions')
        return path
    if lock.exists():raise RuntimeError(f'Hash lock without predictions: {lock}')
    if partial.exists():
        backup=readjson(partial)
        assert backup['protocol_sha256']==PROTO_HASH and backup['weights_sha256']==expected_wsha
        done=int(backup['last_image_id'])
        assert 0<=done<=373
        rows=backup['predictions']
        assert all(int(x['image_id'])<=done for x in rows)
        print(f'Resuming {stem} after image {done} of 373')
    else:
        done=0;rows=[]
    device='cuda:0'
    if family=='YOLO11n-seg':
        model=YOLO(str(WEIGHTS[(family,seed)]))
        assert model.task=='segment'
    else:
        model=new_maskrcnn().to(device)
        checkpoint=torch.load(WEIGHTS[(family,seed)],map_location='cpu',weights_only=False)
        model.load_state_dict(checkpoint['model'],strict=True)
        del checkpoint
        model.eval()
    start=time.perf_counter()
    for iid in range(done+1,374):
        p=paths[iid-1];im=gt_data['images'][iid-1]
        ow,oh=int(im['width']),int(im['height'])
        if family=='YOLO11n-seg':
            result=model.predict(source=str(p),device=0,imgsz=640,conf=0.001,iou=0.7,
                                 max_det=100,retina_masks=True,verbose=False,save=False)[0]
            scores=result.boxes.conf.detach().cpu().numpy() if result.boxes is not None else np.empty(0)
            if len(scores):
                assert result.masks is not None and result.masks.data is not None
                masks=result.masks.data.detach().cpu().numpy()
                assert masks.shape==(len(scores),oh,ow),(iid,masks.shape,(len(scores),oh,ow))
                for s,m in zip(scores,masks):
                    binary=(m>0.5).astype(np.uint8)
                    if binary.any():rows.append({'image_id':iid,'category_id':1,'segmentation':mask_to_rle(binary),'score':float(s)})
        else:
            with Image.open(p) as image:rgb=np.asarray(image.convert('RGB')).copy()
            assert rgb.shape[:2]==(oh,ow)
            with torch.inference_mode():out=model([to_tensor(rgb).to(device)])[0]
            scores=out['scores'].detach().cpu().numpy()
            masks=out['masks'].detach().cpu().numpy()[:,0] if len(scores) else np.zeros((0,oh,ow))
            assert masks.shape==(len(scores),oh,ow)
            for s,m in zip(scores,masks):
                binary=(m>0.5).astype(np.uint8)
                if binary.any():rows.append({'image_id':iid,'category_id':1,'segmentation':mask_to_rle(binary),'score':float(s)})
        if iid%25==0 or iid==373:
            write_atomic(partial,{'protocol_sha256':PROTO_HASH,'weights_sha256':expected_wsha,
                   'last_image_id':iid,'predictions':rows})
            print(f'{stem}: {iid}/373 images, {len(rows)} predictions')
    validate_rows(rows)
    write_atomic(path,rows)
    write_atomic(lock,{'protocol_sha256':PROTO_HASH,'weights_sha256':expected_wsha,
                       'predictions_sha256':sha256(path),'n_predictions':len(rows),
                       'finished_utc':datetime.now(timezone.utc).isoformat(),
                       'runtime_seconds_this_session':round(time.perf_counter()-start,2)})
    if partial.exists():partial.unlink()
    del model;gc.collect();torch.cuda.empty_cache()
    print(f'Completed {stem}: {len(rows)} mask predictions.')
    return path

for family in ('YOLO11n-seg','Mask R-CNN'):
    for seed in (2027,2028):
        PRED_ARCHIVE[(family,seed)]=new_predictions(family,seed)
assert len(PRED_ARCHIVE)==9
print('All 9 archived COCO prediction sets available.')

## 6. Recompute all nine COCO AP values and verify the two historical reports

Reevaluation uses the **same single COCO ground-truth object** and same `maxDets = [1,10,100]` across all nine runs. The five historical AP results must reproduce exactly to rounding precision, or the notebook stops: it must never silently change historical benchmarks.

In [ ]:
gt=COCO();gt.dataset=gt_data
with redirect_stdout(io.StringIO()):gt.createIndex()

def load_prediction_coco(gt,rows):
    # pycocotools.loadRes modifies prediction rows; always give it clean copies.
    if rows:
        safe=[{'image_id':int(r['image_id']),'category_id':int(r['category_id']),
               'segmentation':r['segmentation'],'score':float(r['score'])} for r in rows]
        with redirect_stdout(io.StringIO()):dt=gt.loadRes(safe)
        return dt
    dt=COCO();dt.dataset={'images':gt.dataset['images'],
            'categories':gt.dataset['categories'],'annotations':[]}
    with redirect_stdout(io.StringIO()):dt.createIndex()
    return dt

def coco_ap(gt,dt,imgids=None):
    ev=COCOeval(gt,dt,'segm');ev.params.maxDets=[1,10,100]
    if imgids is not None:ev.params.imgIds=sorted(map(int,imgids))
    with redirect_stdout(io.StringIO()):ev.evaluate();ev.accumulate();ev.summarize()
    return {'Mask_AP50_95':float(ev.stats[0]),'Mask_AP50':float(ev.stats[1]),'Mask_AP75':float(ev.stats[2])}

old09=readjson(REPORT09)
assert old09['protocolo']==ref09
old23=readjson(REPORT23)
assert old23['protocolo']==ref23
expected={}
for record in old09['global']:
    k=('YOLO11n-seg' if 'YOLO' in record['modelo'] else 'Mask R-CNN',2026)
    expected[k]=float(record['mAP50_95_mascaras'])
for record in old23['global']:
    expected[('YOLO26n-seg',int(record['semilla']))]=float(record['mAP50_95_mascaras'])
assert len(expected)==5

records=[];per_date=[];pred_provenance=[]
for family in ('YOLO11n-seg','YOLO26n-seg','Mask R-CNN'):
    for seed in SEEDS:
        pred_path=PRED_ARCHIVE[(family,seed)]
        rows=readjson(pred_path);validate_rows(rows)
        dt=load_prediction_coco(gt,rows)
        gl=coco_ap(gt,dt)
        if (family,seed) in expected:
            assert abs(gl['Mask_AP50_95']-expected[(family,seed)])<=0.00002,(
                'Historical COCO result differs; stop to audit',family,seed,gl,expected[(family,seed)])
        records.append({'Model':family,'Seed':seed,'Images':373,'GT_instances':4648,
                        'Predicted_instances':len(rows),**gl})
        for date,ids in sorted(bydate.items()):
            per_date.append({'Model':family,'Seed':seed,'Date':date,'Images':len(ids),
                             **coco_ap(gt,dt,ids)})
        pred_provenance.append({'Model':family,'Seed':seed,'Predictions':str(pred_path),
                                'Pred_SHA256':sha256(pred_path),
                                'Source':('Archive 09' if (family,seed) in [('YOLO11n-seg',2026),('Mask R-CNN',2026)]
                                          else 'Archive 23' if family=='YOLO26n-seg' else 'New 33')})
        print(f'{family} seed={seed} test mask AP50:95={gl["Mask_AP50_95"]:.5f}')

df_global=pd.DataFrame(records)
df_date=pd.DataFrame(per_date)
assert len(df_global)==9 and len(df_date)==45
assert all(df_global.groupby('Model').size()==3)
display(df_global.round(5))
print('Historical five COCO test results reproduced; nine checkpoints scored.')

## 7. Scientific summary: variation by seed, comparison of model configurations and acquisition dates

The table reports **means and sample standard deviations across three seeds**. This quantifies variation across these trained configurations; it does not infer population-level uncertainty or test equivalence. Date-specific AP is computed with COCO over each date subset; it is **not averaged to obtain the global AP**.

In [ ]:
metric_keys=('Mask_AP50_95','Mask_AP50','Mask_AP75')
summary=[]
for family in ('YOLO11n-seg','YOLO26n-seg','Mask R-CNN'):
    f=df_global[df_global.Model==family]
    d={'Model':family,'Seeds':3}
    for m in metric_keys:
        v=f[m].to_numpy(dtype=float)
        d[m+'_mean']=float(v.mean())
        d[m+'_sd_sample']=float(v.std(ddof=1))
        d[m+'_min']=float(v.min());d[m+'_max']=float(v.max())
    summary.append(d)
df_summary=pd.DataFrame(summary)
# Differences are purely descriptive differences in seed means: no paired equivalence p-values.
diffs=[]
for a,b in [('YOLO11n-seg','Mask R-CNN'),('YOLO26n-seg','Mask R-CNN'),('YOLO11n-seg','YOLO26n-seg')]:
    ar=df_summary.set_index('Model').loc[a];br=df_summary.set_index('Model').loc[b]
    diffs.append({'Comparison':f'{a} minus {b}',
                  **{m+'_mean_difference':float(ar[m+'_mean']-br[m+'_mean']) for m in metric_keys},
                  'Interpretation':'Descriptive mean difference across 3 seeds; not a significance or equivalence test.'})
df_diffs=pd.DataFrame(diffs)
df_checkpoints=pd.DataFrame(metadata).merge(pd.DataFrame(pred_provenance),on=['Model','Seed'])
df_limits=pd.DataFrame({'Limitation':[
    'The FaroPigSeg test set was consulted in earlier experiments; this is a post-hoc expanded descriptive analysis.',
    'Only five acquisition-date groups, with uncertain independence among animals, cameras and sessions.',
    'Three random seeds per architecture do not create independent farms or solve the low number of date clusters.',
    'YOLO11/YOLO26 and Mask R-CNN have different architectures, input resolutions, batch sizes and epoch budgets.',
    'COCO segmentation AP is calculated from the same 373-image GT. No new checkpoint selection used test.',
    'No claims of external generalization, occlusion robustness, tracking or on-farm deployment.',
    'Validation thresholds for counting and confidence calibration are separate analyses, not optimized here.',
    'Device-specific training/latency measurements are not a fair hardware-normalized comparison.',
]})
print('TEST summary (mean ± sample SD, n=3):')
display(df_summary[['Model','Mask_AP50_95_mean','Mask_AP50_95_sd_sample']].round(5))
display(df_diffs[['Comparison','Mask_AP50_95_mean_difference']].round(5))

## 8. Publication figures — English, 600 DPI and vector formats

Figure A shows nine individual test AP values, plus per-architecture mean; Figure B displays per-date mean AP across three seeds (descriptive only). Both figures have readable captions, not artificial significance markings.

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,
   'axes.titlesize':12,'axes.labelsize':10,'xtick.labelsize':10,'ytick.labelsize':10})
MODELS=['YOLO11n-seg','YOLO26n-seg','Mask R-CNN']
COLORS={'YOLO11n-seg':'#275A81','YOLO26n-seg':'#3C9175','Mask R-CNN':'#B16C41'}

fig,ax=plt.subplots(figsize=(8.7,4.6))
for pos,m in enumerate(MODELS):
    subset=df_global[df_global.Model==m].sort_values('Seed')
    x=np.array([pos-.15,pos,pos+.15])
    ax.scatter(x,subset['Mask_AP50_95'],s=70,color=COLORS[m],edgecolor='white',linewidth=0.8,zorder=3)
    for xx,row in zip(x,subset.itertuples()):
        ax.text(xx,float(row.Mask_AP50_95)+0.006,str(row.Seed),ha='center',fontsize=8)
    mn=float(subset['Mask_AP50_95'].mean());ax.plot([pos-.25,pos+.25],[mn,mn],color='black',lw=1.6)
ax.set_xticks(range(3),MODELS)
ax.set_ylim(max(0,float(df_global['Mask_AP50_95'].min())-0.055),min(1,float(df_global['Mask_AP50_95'].max())+0.055))
ax.set_ylabel('COCO mask AP@[0.50:0.95] — test')
ax.set_title('Instance-segmentation accuracy across three training seeds')
ax.grid(axis='y',alpha=.22)
fig.tight_layout()
for ext in ('png','pdf','svg'):fig.savefig(FIGS/f'CERDOS_33_AP_9_Seeds.{ext}',dpi=600,bbox_inches='tight')
plt.show();plt.close(fig)

dates=sorted(bydate)
mean_date=(df_date.groupby(['Model','Date'])['Mask_AP50_95'].mean().unstack('Date').loc[MODELS,dates])
fig,ax=plt.subplots(figsize=(9.1,3.7))
lo=float(mean_date.min().min());hi=float(mean_date.max().max())
image=ax.imshow(mean_date.to_numpy(),cmap='viridis',aspect='auto',vmin=max(0,lo-.03),vmax=min(1,hi+.03))
ax.set_xticks(range(len(dates)),dates,rotation=30,ha='right')
ax.set_yticks(range(3),MODELS)
ax.set_title('Mean date-specific mask AP across three seeds (descriptive)')
for i in range(3):
    for j in range(len(dates)):
        v=float(mean_date.iloc[i,j]);ax.text(j,i,f'{v:.3f}',ha='center',va='center',
                color='white' if v<(lo+hi)/2 else 'black',fontsize=10)
fig.colorbar(image,ax=ax,label='Date-specific mask AP@[0.50:0.95]',shrink=.8)
fig.tight_layout()
for ext in ('png','pdf','svg'):fig.savefig(FIGS/f'CERDOS_33_AP_By_Date.{ext}',dpi=600,bbox_inches='tight')
plt.show();plt.close(fig)

## 9. Auditable Excel, provenance report and downloadable ZIP

The workbook separates per-seed AP, per-date AP, mean/SD summaries, descriptive comparisons, SHA-256 provenance and limitations. **COCO prediction JSON files remain in Drive** under Experiments 09, 23 and 33; they can later be linked to a public reproducibility archive subject to dataset permissions.

In [ ]:
EXCEL=METRICS/'CERDOS_33_COCO_9_Semillas.xlsx'
REPORT_FILE=OUT/'CERDOS_33_Informe_Final.json'
REPORT={
 'experiment':33,'created_utc':datetime.now(timezone.utc).isoformat(),
 'journal_target':'Smart Agricultural Technology (Elsevier)',
 'analysis':'Harmonized COCO instance mask AP over nine existing trained checkpoints',
 'protocol':PROTO,'protocol_sha256':PROTO_HASH,
 'software':{'python':platform.python_version(),'torch':torch.__version__,
             'torchvision':torchvision.__version__,'ultralytics':ultralytics.__version__,
             'gpu':torch.cuda.get_device_name(0)},
 'global':records,'per_date':per_date,'seed_summary':summary,'descriptive_differences':diffs,
 'predictions_provenance':pred_provenance,
 'checkpoint_sha256':{str((m,s)):sha256(p) for (m,s),p in WEIGHTS.items()},
 'limitations':df_limits.Limitation.tolist(),
 'test_is_new_independent_validation':False,
 'inference_on_new_checkpoints_only':True,
}
with pd.ExcelWriter(EXCEL,engine='openpyxl') as writer:
    for name,df in [('Global_9',df_global),('Date_45',df_date),('Seed_Summary',df_summary),
                    ('Mean_Differences',df_diffs),('Checkpoints',df_checkpoints),
                    ('Protocol',pd.DataFrame([{'Field':k,'Value':str(v)} for k,v in PROTO.items()])),
                    ('Limitations',df_limits)]:
        df.to_excel(writer,sheet_name=name,index=False)
        ws=writer.sheets[name]
        from openpyxl.styles import Alignment, Font, PatternFill
        ws.freeze_panes='A2';ws.auto_filter.ref=ws.dimensions
        for cell in ws[1]:
            cell.font=Font(name='Calibri',size=11,bold=True,color='FFFFFF')
            cell.fill=PatternFill('solid',fgColor='193750')
            cell.alignment=Alignment(wrap_text=True,vertical='center')
        ws.row_dimensions[1].height=28
        for col in ws.columns:
            header=str(col[0].value or '')
            width=min(48,max(13,len(header)+3))
            if header in ('Model','Date','Comparison'):width=38 if header=='Comparison' else 22
            if header=='Limitation':width=100
            if header in ('Checkpoint','Predictions'):width=60
            if header in ('SHA256','Pred_SHA256'):width=68
            if header in ('Value','Field'):width=55 if header=='Value' else 34
            ws.column_dimensions[col[0].column_letter].width=width
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value,float):cell.number_format='0.00000'
write_atomic(REPORT_FILE,REPORT)
ZIP=EXPORT/'CERDOS_33_COCO_9_Semillas_Resultados.zip'
OUTFILES=[EXCEL,REPORT_FILE,PROTO_FILE]
for name in ('CERDOS_33_AP_9_Seeds','CERDOS_33_AP_By_Date'):
    OUTFILES.extend(FIGS/f'{name}.{ext}' for ext in ('png','pdf','svg'))
for fam,seed in [('YOLO11n-seg',2027),('YOLO11n-seg',2028),('Mask R-CNN',2027),('Mask R-CNN',2028)]:
    slug='YOLO11' if fam=='YOLO11n-seg' else 'MaskRCNN'
    OUTFILES.append(OUT/f'CERDOS_33_Pred_{slug}_seed_{seed}_hash.json')
assert all(p.is_file() for p in OUTFILES)
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in OUTFILES:z.write(p,arcname=p.name)
with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None
print('Workbook:',EXCEL)
print('COCO provenance report:',REPORT_FILE)
print('ZIP:',ZIP,'| items:',len(OUTFILES),'| MB:',round(ZIP.stat().st_size/1e6,2))
files.download(str(ZIP))

## Scientific interpretation and next manuscript step

- **What this completes:** common-test mask AP (COCO global and by date) for 3 architectures × 3 pre-existing trained seeds, with SHA-256-tracked predictions and reproducible exports.
- **What it does not establish:** a fresh blind test, between-farm generalization, causal superiority of an architecture, hardware-normalized efficiency, tracking, weight prediction, or statistical equivalence.
- **Statistical plan:** report effects and the three observed seed values (mean ± sample SD), with five acquisition dates treated as descriptive clusters; do not interpret 45 architecture–seed–date entries as 45 independent farms. Bootstrap intervals over five date clusters, if presented, are exploratory and must be explicitly qualified.
- **Next step:** review `CERDOS_33_COCO_9_Semillas_Resultados.zip`, update the Elsevier *Smart Agricultural Technology* paper with the verified nine-run test results and finalize reproducibility, title, references, authors' declarations and license/permission statements. **No more model training is prescribed by this notebook.**